In [ ]:
# -*- coding: utf-8 -*-
"""Improved NFR Classification Model - Anti-Overfitting Edition"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, accuracy_score,
                            precision_recall_fscore_support, confusion_matrix,
                            f1_score)
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm
from torch.optim import AdamW
import os
import json
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [ ]:

# ============================================================================
# ENHANCED CONFIGURATION - ANTI-OVERFITTING
# ============================================================================
class Config:
    # Paths
    DATA_PATH = "/content/drive/MyDrive/fyp_data/ISO_NFR_dataset.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/NFR_Model2"
    CHECKPOINT_DIR = "/content/drive/MyDrive/NFRCheckpoints2"

    # 7 NFR Classes
    QUALITY_ATTRIBUTES = [
        'Performance Efficiency', 'Compatibility', 'Usability',
        'Reliability', 'Security', 'Maintainability', 'Portability'
    ]

    # Model Configuration
    MODEL_NAME = "roberta-base"
    MAX_LENGTH = 128
    DROPOUT_RATE = 0.5  # Increased from 0.4 for stronger regularization

    # Training Configuration - ANTI-OVERFITTING
    BATCH_SIZE = 24
    NUM_EPOCHS = 15
    LEARNING_RATE = 1e-5  # Reduced from 1.5e-5 for slower, safer learning
    WEIGHT_DECAY = 0.03  # Increased from 0.02 for stronger L2 regularization
    GRADIENT_CLIP_VALUE = 0.5
    GRADIENT_ACCUMULATION_STEPS = 2
    LABEL_SMOOTHING = 0.1  # NEW: Reduces overconfidence

    # Data Split
    VAL_SIZE = 0.15
    TEST_SIZE = 0.15

    # Training Strategy
    USE_CLASS_WEIGHTS = True
    USE_AMP = True  # Mixed precision training
    USE_MIXUP = True  # NEW: Mixup data augmentation
    MIXUP_ALPHA = 0.2  # Mixup parameter

    # Early Stopping - STRICTER
    EARLY_STOPPING_PATIENCE = 3  # Reduced from 4
    MIN_DELTA = 0.002  # Increased from 0.001 - require bigger improvement
    OVERFITTING_THRESHOLD = 0.08  # Reduced from 0.15 - stricter check

    RANDOM_STATE = 42

config = Config()

# Create directories
for directory in [config.OUTPUT_DIR, config.CHECKPOINT_DIR]:
    os.makedirs(directory, exist_ok=True)

print("="*80)
print("NFR CLASSIFICATION MODEL - ANTI-OVERFITTING EDITION")
print("="*80)
print(f"🛡️ Dropout: {config.DROPOUT_RATE}")
print(f"🛡️ Learning Rate: {config.LEARNING_RATE}")
print(f"🛡️ Weight Decay: {config.WEIGHT_DECAY}")
print(f"🛡️ Label Smoothing: {config.LABEL_SMOOTHING}")
print(f"🛡️ Mixup: {'Enabled' if config.USE_MIXUP else 'Disabled'}")
print(f"🛡️ Overfitting Threshold: {config.OVERFITTING_THRESHOLD}")
print("="*80)


NFR CLASSIFICATION MODEL - ANTI-OVERFITTING EDITION
🛡️ Dropout: 0.5
🛡️ Learning Rate: 1e-05
🛡️ Weight Decay: 0.03
🛡️ Label Smoothing: 0.1
🛡️ Mixup: Enabled
🛡️ Overfitting Threshold: 0.08


In [ ]:
# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================
class EnhancedRobertaClassifier(nn.Module):
    """Enhanced RoBERTa with strong regularization"""

    def __init__(self, num_classes, model_name='roberta-base', dropout_rate=0.5):
        super(EnhancedRobertaClassifier, self).__init__()

        self.roberta = RobertaModel.from_pretrained(model_name)
        self.hidden_size = self.roberta.config.hidden_size

        # Enhanced classification head with high dropout
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dense1 = nn.Linear(self.hidden_size, self.hidden_size)
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)

        self.dropout2 = nn.Dropout(dropout_rate)
        self.dense2 = nn.Linear(self.hidden_size, 256)
        self.layer_norm2 = nn.LayerNorm(256)

        self.dropout3 = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(256, num_classes)

        self._init_weights()

    def _init_weights(self):
        for module in [self.dense1, self.dense2, self.classifier]:
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output

        x = self.dropout1(pooled_output)
        x = self.dense1(x)
        x = self.layer_norm1(x)
        x = torch.relu(x)

        x = self.dropout2(x)
        x = self.dense2(x)
        x = self.layer_norm2(x)
        x = torch.relu(x)

        x = self.dropout3(x)
        logits = self.classifier(x)

        return logits

In [ ]:

# ============================================================================
# DATASET CLASS
# ============================================================================
class NFRDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# ============================================================================
# IMPROVED EARLY STOPPING CLASS
# ============================================================================
class EarlyStopping:
    """Enhanced early stopping with overfitting detection"""

    def __init__(self, patience=3, min_delta=0.002, overfitting_threshold=0.08):
        self.patience = patience
        self.min_delta = min_delta
        self.overfitting_threshold = overfitting_threshold
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0

    def __call__(self, val_f1, train_loss, val_loss, epoch):
        score = val_f1
        loss_gap = val_loss - train_loss

        # Check for overfitting
        if loss_gap > self.overfitting_threshold:
            print(f"⚠️ Overfitting detected: val_loss ({val_loss:.4f}) >> train_loss ({train_loss:.4f}), gap={loss_gap:.4f}")
            self.counter += 1

        # Check for improvement
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            print(f"⏳ No significant improvement for {self.counter}/{self.patience} epochs (improvement < {self.min_delta})")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"🛑 Early stopping triggered!")
        else:
            improvement = score - self.best_score
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            print(f"✨ New best F1: {score:.4f} (↑{improvement:.4f})")

        return self.early_stop

In [ ]:
# ============================================================================
# MANIFOLD MIXUP - Applied to embeddings, not input_ids
# ============================================================================
def manifold_mixup(embeddings, labels, alpha=0.2):
    """
    Apply mixup to embeddings (not input_ids).
    This works because embeddings are continuous, unlike discrete token IDs.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = embeddings.size()[0]
    index = torch.randperm(batch_size).to(embeddings.device)

    mixed_embeddings = lam * embeddings + (1 - lam) * embeddings[index]
    y_a, y_b = labels, labels[index]
    return mixed_embeddings, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Calculate loss for mixup"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [ ]:
# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

try:
    df = pd.read_csv(config.DATA_PATH, encoding='utf-8')
except UnicodeDecodeError:
    print("⚠️ UTF-8 decoding failed, trying latin-1...")
    df = pd.read_csv(config.DATA_PATH, encoding='latin-1')

df.columns = df.columns.str.strip()
df = df.rename(columns={'Requirement': 'requirement_text', 'type': 'label'})
df['label'] = df['label'].str.strip()

print("\n📊 Class Distribution:")
for label, count in df['label'].value_counts().items():
    print(f"  {label}: {count}")

# Encode labels
label_encoder = LabelEncoder()
df['encoded_label'] = label_encoder.fit_transform(df['label'])

# Save label mapping
label_mapping = {
    label: int(value)
    for label, value in zip(label_encoder.classes_,
                           label_encoder.transform(label_encoder.classes_))
}

with open(os.path.join(config.OUTPUT_DIR, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping, f, indent=4)

print(f"\n✅ Total samples: {len(df)}")
print(f"✅ Number of classes: {len(label_encoder.classes_)}")
print(f"✅ Classes: {', '.join(label_encoder.classes_)}")


📦 Loading tokenizer and label mapping...
✅ Loaded 7 labels: ['Compatibility', 'Maintainability', 'Performance', 'Portability', 'Reliability', 'Security', 'Usability']
📦 Loading full model directly...


AttributeError: 'dict' object has no attribute 'to'

In [ ]:
# ============================================================================
# STRATIFIED TRAIN-VAL-TEST SPLIT
# ============================================================================
print("\n" + "="*80)
print("CREATING TRAIN-VAL-TEST SPLITS")
print("="*80)

X_temp, X_test, y_temp, y_test = train_test_split(
    df['requirement_text'], df['encoded_label'],
    test_size=config.TEST_SIZE, stratify=df['encoded_label'],
    random_state=config.RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=config.VAL_SIZE/(1-config.TEST_SIZE),
    stratify=y_temp, random_state=config.RANDOM_STATE
)

print(f"✅ Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Define confusion matrix for similar classes
CONFUSING_PAIRS = {
    'Compatibility': ['Portability'],
    'Portability': ['Compatibility'],
    'Security': ['Reliability'],
    'Reliability': ['Security']
}

print(f"\n⚠️ Monitoring confusing class pairs:")
for class_a, confusing_classes in CONFUSING_PAIRS.items():
    print(f"  {class_a} ↔ {', '.join(confusing_classes)}")

# Calculate class weights
if config.USE_CLASS_WEIGHTS:
    class_counts = Counter(y_train)
    total_samples = len(y_train)
    num_classes = len(label_encoder.classes_)
    class_weights = torch.tensor(
        [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)],
        dtype=torch.float
    )
    print(f"\n📊 Class weights: {class_weights.numpy()}")
else:
    class_weights = None



CREATING TRAIN-VAL-TEST SPLITS
✅ Train: 4824 | Val: 1034 | Test: 1034

⚠️ Monitoring confusing class pairs:
  Compatibility ↔ Portability
  Portability ↔ Compatibility
  Security ↔ Reliability
  Reliability ↔ Security

📊 Class weights: [1.0075188  1.008994   0.97612303 0.97199273 0.9788961  1.0938776
 0.9733656 ]


In [ ]:
# ============================================================================
# TOKENIZATION AND DATA LOADERS
# ============================================================================
print("\n" + "="*80)
print("CREATING DATA LOADERS")
print("="*80)

tokenizer = RobertaTokenizer.from_pretrained(config.MODEL_NAME)

train_dataset = NFRDataset(X_train, y_train, tokenizer, config.MAX_LENGTH)
val_dataset = NFRDataset(X_val, y_val, tokenizer, config.MAX_LENGTH)
test_dataset = NFRDataset(X_test, y_test, tokenizer, config.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)

print("✅ Data loaders created successfully!")


CREATING DATA LOADERS


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

✅ Data loaders created successfully!


In [ ]:
# ============================================================================
# MODEL INITIALIZATION
# ============================================================================
print("\n" + "="*80)
print("INITIALIZING MODEL")
print("="*80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

num_classes = len(label_encoder.classes_)
model = EnhancedRobertaClassifier(
    num_classes=num_classes,
    model_name=config.MODEL_NAME,
    dropout_rate=config.DROPOUT_RATE
)
model.to(device)

# Loss function with label smoothing
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device) if class_weights is not None else None,
    label_smoothing=config.LABEL_SMOOTHING
)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# Mixed precision scaler
scaler = GradScaler() if config.USE_AMP else None

print(f"✅ Model initialized with {num_classes} classes")
print(f"📊 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"📊 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")



INITIALIZING MODEL
🖥️ Using device: cuda


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model initialized with 7 classes
📊 Total parameters: 125,436,935
📊 Trainable parameters: 125,436,935


In [ ]:
# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_epoch(model, train_loader, criterion, optimizer, device, scaler, epoch):
    """Training loop with mixup augmentation"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Apply mixup for regularization (50% chance)
        use_mixup = config.USE_MIXUP and np.random.rand() < 0.5

        if config.USE_AMP:
            with autocast():
                # Get embeddings from RoBERTa before the classification head
                outputs = model.roberta(input_ids=input_ids, attention_mask=attention_mask)
                pooled_output = outputs.pooler_output

                if use_mixup:
                    # Apply mixup to the pooled embeddings
                    mixed_embeddings, labels_a, labels_b, lam = manifold_mixup(pooled_output, labels, config.MIXUP_ALPHA)
                    # Pass the mixed embeddings through the rest of the model
                    x = model.dropout1(mixed_embeddings)
                    x = model.dense1(x)
                    x = model.layer_norm1(x)
                    x = torch.relu(x)

                    x = model.dropout2(x)
                    x = model.dense2(x)
                    x = model.layer_norm2(x)
                    x = torch.relu(x)

                    x = model.dropout3(x)
                    logits = model.classifier(x)

                    loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
                else:
                    # Pass the original embeddings through the rest of the model
                    x = model.dropout1(pooled_output)
                    x = model.dense1(x)
                    x = model.layer_norm1(x)
                    x = torch.relu(x)

                    x = model.dropout2(x)
                    x = model.dense2(x)
                    x = model.layer_norm2(x)
                    x = torch.relu(x)

                    x = model.dropout3(x)
                    logits = model.classifier(x)

                    loss = criterion(logits, labels)

                loss = loss / config.GRADIENT_ACCUMULATION_STEPS

            scaler.scale(loss).backward()

            if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            # Get embeddings from RoBERTa before the classification head
            outputs = model.roberta(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.pooler_output

            if use_mixup:
                # Apply mixup to the pooled embeddings
                mixed_embeddings, labels_a, labels_b, lam = manifold_mixup(pooled_output, labels, config.MIXUP_ALPHA)
                # Pass the mixed embeddings through the rest of the model
                x = model.dropout1(mixed_embeddings)
                x = model.dense1(x)
                x = model.layer_norm1(x)
                x = torch.relu(x)

                x = model.dropout2(x)
                x = model.dense2(x)
                x = model.layer_norm2(x)
                x = torch.relu(x)

                x = model.dropout3(x)
                logits = model.classifier(x)

                loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
            else:
                # Pass the original embeddings through the rest of the model
                x = model.dropout1(pooled_output)
                x = model.dense1(x)
                x = model.layer_norm1(x)
                x = torch.relu(x)

                x = model.dropout2(x)
                x = model.dense2(x)
                x = model.layer_norm2(x)
                x = torch.relu(x)

                x = model.dropout3(x)
                logits = model.classifier(x)

                loss = criterion(logits, labels)

            loss = loss / config.GRADIENT_ACCUMULATION_STEPS
            loss.backward()

            if (batch_idx + 1) % config.GRADIENT_ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP_VALUE)
                optimizer.step()
                optimizer.zero_grad()

        # Calculate accuracy (using original labels for mixup)
        _, predicted = torch.max(logits, 1)
        if use_mixup:
            total += labels_a.size(0)
            correct += (predicted == labels_a).sum().item()
        else:
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        total_loss += loss.item() * config.GRADIENT_ACCUMULATION_STEPS

        progress_bar.set_postfix({
            'loss': f'{loss.item() * config.GRADIENT_ACCUMULATION_STEPS:.4f}',
            'acc': f'{100. * correct / total:.2f}%'
        })

    return total_loss / len(train_loader), correct / total

def validate(model, val_loader, criterion, device):
    """Validation loop"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            if config.USE_AMP:
                with autocast():
                    logits = model(input_ids, attention_mask)
                    loss = criterion(logits, labels)
            else:
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)

            total_loss += loss.item()
            _, predicted = torch.max(logits, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )

    return {
        'loss': total_loss / len(val_loader),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

In [ ]:
# ============================================================================
# TRAINING LOOP WITH ENHANCED EARLY STOPPING
# ============================================================================
print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)

early_stopping = EarlyStopping(
    patience=config.EARLY_STOPPING_PATIENCE,
    min_delta=config.MIN_DELTA,
    overfitting_threshold=config.OVERFITTING_THRESHOLD
)

best_val_f1 = 0
training_history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_f1': []
}

for epoch in range(config.NUM_EPOCHS):
    print(f"\n{'='*80}")
    print(f"EPOCH {epoch + 1}/{config.NUM_EPOCHS}")
    print(f"{'='*80}")

    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, scaler, epoch + 1
    )

    # Validate
    val_metrics = validate(model, val_loader, criterion, device)

    # Update scheduler
    scheduler.step(val_metrics['f1'])

    # Store history
    training_history['train_loss'].append(train_loss)
    training_history['train_acc'].append(train_acc)
    training_history['val_loss'].append(val_metrics['loss'])
    training_history['val_acc'].append(val_metrics['accuracy'])
    training_history['val_f1'].append(val_metrics['f1'])

    # Print results
    print(f"\n📊 Results:")
    print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"  Val: Loss={val_metrics['loss']:.4f}, Acc={val_metrics['accuracy']:.4f}, F1={val_metrics['f1']:.4f}")

    # Show overfitting indicator
    loss_gap = val_metrics['loss'] - train_loss
    if loss_gap > 0.05:
        print(f"  ⚠️ Loss gap: {loss_gap:.4f} (potential overfitting)")
    else:
        print(f"  ✅ Loss gap: {loss_gap:.4f} (healthy)")

    # Save best model
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_model_path = os.path.join(config.OUTPUT_DIR, "best_model.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_f1': val_metrics['f1'],
            'label_encoder': label_encoder,
            'config': {
                'model_name': config.MODEL_NAME,
                'num_classes': num_classes,
                'dropout_rate': config.DROPOUT_RATE,
                'max_length': config.MAX_LENGTH
            }
        }, best_model_path)

    # Check early stopping
    if early_stopping(val_metrics['f1'], train_loss, val_metrics['loss'], epoch + 1):
        print(f"\n🛑 Training stopped at epoch {epoch + 1}")
        print(f"🏆 Best F1: {early_stopping.best_score:.4f} at epoch {early_stopping.best_epoch}")
        break



STARTING TRAINING

EPOCH 1/15


Epoch 1: 100%|██████████| 201/201 [00:30<00:00,  6.67it/s, loss=1.7667, acc=17.83%]



📊 Results:
  Train: Loss=1.9450, Acc=0.1783
  Val: Loss=1.5250, Acc=0.5870, F1=0.5620
  ✅ Loss gap: -0.4200 (healthy)

EPOCH 2/15


Epoch 2: 100%|██████████| 201/201 [00:31<00:00,  6.30it/s, loss=1.2763, acc=49.79%]



📊 Results:
  Train: Loss=1.5322, Acc=0.4979
  Val: Loss=0.9319, Acc=0.8994, F1=0.8999
  ✅ Loss gap: -0.6004 (healthy)
✨ New best F1: 0.8999 (↑0.3378)

EPOCH 3/15


...


📊 Results:
  Train: Loss=0.6585, Acc=0.8141
  Val: Loss=0.5937, Acc=0.9652, F1=0.9652
  ✅ Loss gap: -0.0648 (healthy)
⏳ No significant improvement for 1/3 epochs (improvement < 0.002)

EPOCH 10/15


Epoch 10: 100%|██████████| 201/201 [00:33<00:00,  6.03it/s, loss=1.0097, acc=77.45%]



📊 Results:
  Train: Loss=0.6322, Acc=0.7745
  Val: Loss=0.6106, Acc=0.9613, F1=0.9613
  ✅ Loss gap: -0.0216 (healthy)
⏳ No significant improvement for 2/3 epochs (improvement < 0.002)

EPOCH 11/15


Epoch 11: 100%|██████████| 201/201 [00:31<00:00,  6.47it/s, loss=0.5105, acc=82.48%]
                                                           


📊 Results:
  Train: Loss=0.5933, Acc=0.8248
  Val: Loss=0.5846, Acc=0.9691, F1=0.9690
  ✅ Loss gap: -0.0087 (healthy)
⏳ No significant improvement for 3/3 epochs (improvement < 0.002)
🛑 Early stopping triggered!

🛑 Training stopped at epoch 11
🏆 Best F1: 0.9710 at epoch 8


In [ ]:
# ============================================================================
# FINAL TEST EVALUATION
# ============================================================================
print("\n" + "="*80)
print("FINAL TEST EVALUATION")
print("="*80)

# Load best model
checkpoint = torch.load(os.path.join(config.OUTPUT_DIR, "best_model.pt"),
                       map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Test evaluation
test_metrics = validate(model, test_loader, criterion, device)

print("\n📊 FINAL TEST RESULTS:")
print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall: {test_metrics['recall']:.4f}")
print(f"  F1 Score: {test_metrics['f1']:.4f}")

# Detailed classification report
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORT")
print("="*80)
print("\n" + classification_report(
    test_metrics['labels'],
    test_metrics['predictions'],
    target_names=label_encoder.classes_,
    digits=4
))

# Analyze specific confusions
print("\n" + "="*80)
print("CONFUSION ANALYSIS")
print("="*80)

cm = confusion_matrix(test_metrics['labels'], test_metrics['predictions'])
class_names = label_encoder.classes_

confusion_found = False
for i, true_class in enumerate(class_names):
    for j, pred_class in enumerate(class_names):
        if i != j and cm[i][j] > 0:
            confusion_rate = cm[i][j] / cm[i].sum() * 100
            if confusion_rate > 5:  # Show if >5% confusion
                print(f"⚠️ {true_class} misclassified as {pred_class}: {cm[i][j]} times ({confusion_rate:.1f}%)")
                confusion_found = True

if not confusion_found:
    print("✅ No significant class confusions detected (all <5%)")

# Save results
final_results = {
    'test_f1': float(test_metrics['f1']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_precision': float(test_metrics['precision']),
    'test_recall': float(test_metrics['recall']),
    'best_epoch': int(checkpoint['epoch']),
    'training_history': training_history,
    'label_mapping': label_mapping
}

with open(os.path.join(config.OUTPUT_DIR, "results.json"), 'w') as f:
    json.dump(final_results, f, indent=4)



FINAL TEST EVALUATION



📊 FINAL TEST RESULTS:
  Accuracy: 0.9691
  Precision: 0.9694
  Recall: 0.9691
  F1 Score: 0.9691

DETAILED CLASSIFICATION REPORT

                 precision    recall  f1-score   support

  Compatibility     1.0000    0.9660    0.9827       147
Maintainability     0.9658    0.9658    0.9658       146
    Performance     0.9799    0.9669    0.9733       151
    Portability     0.9548    0.9737    0.9642       152
    Reliability     0.9732    0.9603    0.9667       151
       Security     0.9635    0.9778    0.9706       135
      Usability     0.9487    0.9737    0.9610       152

       accuracy                         0.9691      1034
      macro avg     0.9694    0.9691    0.9692      1034
   weighted avg     0.9694    0.9691    0.9691      1034


CONFUSION ANALYSIS
✅ No significant class confusions detected (all <5%)


In [ ]:
# ============================================================================
# VISUALIZATIONS (CONFUSION MATRIX + LOSS GRAPH ONLY)
# ============================================================================
print("\n" + "="*80)
print("GENERATING VISUALIZATIONS")
print("="*80)

# 1. Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - NFR Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "confusion_matrix.png"),
            dpi=300, bbox_inches='tight')
print("✅ Confusion matrix saved")
plt.close()

# 2. Loss Graph
plt.figure(figsize=(10, 6))
epochs_range = range(1, len(training_history['train_loss']) + 1)
plt.plot(epochs_range, training_history['train_loss'],
         label='Train Loss', marker='o', linewidth=2, markersize=6)
plt.plot(epochs_range, training_history['val_loss'],
         label='Validation Loss', marker='s', linewidth=2, markersize=6)

# Add overfitting threshold line
max_train_loss = max(training_history['train_loss'])
plt.axhline(y=max_train_loss + config.OVERFITTING_THRESHOLD,
            color='r', linestyle='--', alpha=0.5,
            label=f'Overfitting Threshold (+{config.OVERFITTING_THRESHOLD})')

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, "loss_graph.png"),
            dpi=300, bbox_inches='tight')
print("✅ Loss graph saved")
plt.close()

# ============================================================================
# TRAINING SUMMARY
# ============================================================================
print("\n" + "="*80)
print("🎉 TRAINING COMPLETED!")
print("="*80)
print(f"\n📂 Results saved to: {config.OUTPUT_DIR}")
print(f"\n📊 Training Summary:")
print(f"  Total Epochs: {len(training_history['train_loss'])}")
print(f"  Best Epoch: {checkpoint['epoch']}")
print(f"  Best Val F1: {checkpoint['val_f1']:.4f}")
print(f"  Final Test F1: {test_metrics['f1']:.4f}")
print(f"  Final Test Accuracy: {test_metrics['accuracy']:.4f}")

# Check if overfitting was avoided
final_loss_gap = training_history['val_loss'][-1] - training_history['train_loss'][-1]
if final_loss_gap < config.OVERFITTING_THRESHOLD:
    print(f"\n✅ SUCCESS: No overfitting detected (loss gap: {final_loss_gap:.4f})")
else:
    print(f"\n⚠️ WARNING: Possible overfitting (loss gap: {final_loss_gap:.4f})")



GENERATING VISUALIZATIONS
✅ Confusion matrix saved
✅ Loss graph saved

🎉 TRAINING COMPLETED!

📂 Results saved to: /content/drive/MyDrive/NFR_Model2

📊 Training Summary:
  Total Epochs: 11
  Best Epoch: 8
  Best Val F1: 0.9710
  Final Test F1: 0.9691
  Final Test Accuracy: 0.9691

✅ SUCCESS: No overfitting detected (loss gap: -0.0087)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
import json
import os

# ============================================================================
# MODEL ARCHITECTURE (Must match training architecture)
# ============================================================================
class EnhancedRobertaClassifier(nn.Module):
    """Enhanced RoBERTa with strong regularization"""

    def __init__(self, num_classes, model_name='roberta-base', dropout_rate=0.5):
        super(EnhancedRobertaClassifier, self).__init__()

        self.roberta = RobertaModel.from_pretrained(model_name)
        self.hidden_size = self.roberta.config.hidden_size

        # Enhanced classification head with high dropout
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dense1 = nn.Linear(self.hidden_size, self.hidden_size)
        self.layer_norm1 = nn.LayerNorm(self.hidden_size)

        self.dropout2 = nn.Dropout(dropout_rate)
        self.dense2 = nn.Linear(self.hidden_size, 256)
        self.layer_norm2 = nn.LayerNorm(256)

        self.dropout3 = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output

        x = self.dropout1(pooled_output)
        x = self.dense1(x)
        x = self.layer_norm1(x)
        x = torch.relu(x)

        x = self.dropout2(x)
        x = self.dense2(x)
        x = self.layer_norm2(x)
        x = torch.relu(x)

        x = self.dropout3(x)
        logits = self.classifier(x)

        return logits


# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    MODEL_PATH = "/content/drive/MyDrive/NFR_Model2"
    MODEL_FILE = "best_model.pt"
    LABEL_MAP_FILE = "label_mapping.json"
    BASE_MODEL = "roberta-base"
    MAX_LENGTH = 128
    DROPOUT_RATE = 0.5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()


# ============================================================================
# LOAD MODEL AND COMPONENTS
# ============================================================================
def load_model():
    """Load the trained model and associated components"""

    print("=" * 80)
    print("LOADING NFR CLASSIFICATION MODEL")
    print("=" * 80)

    # Load label mapping
    label_map_path = os.path.join(config.MODEL_PATH, config.LABEL_MAP_FILE)
    with open(label_map_path, 'r') as f:
        label_mapping = json.load(f)

    # Create reverse mapping (index -> label)
    id2label = {int(v): k for k, v in label_mapping.items()}
    num_classes = len(label_mapping)

    print(f"✅ Loaded {num_classes} classes: {', '.join(label_mapping.keys())}")

    # Load tokenizer
    tokenizer = RobertaTokenizer.from_pretrained(config.BASE_MODEL)
    print(f"✅ Loaded tokenizer: {config.BASE_MODEL}")

    # Initialize model architecture
    model = EnhancedRobertaClassifier(
        num_classes=num_classes,
        model_name=config.BASE_MODEL,
        dropout_rate=config.DROPOUT_RATE
    )

    # Load checkpoint
    checkpoint_path = os.path.join(config.MODEL_PATH, config.MODEL_FILE)
    checkpoint = torch.load(checkpoint_path, map_location=config.DEVICE, weights_only=False)

    # Load model weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(config.DEVICE)
    model.eval()

    print(f"✅ Loaded model from epoch {checkpoint['epoch']} with Val F1: {checkpoint['val_f1']:.4f}")
    print(f"🖥️ Using device: {config.DEVICE}")
    print("=" * 80)

    return model, tokenizer, id2label


# ============================================================================
# PREDICTION FUNCTION
# ============================================================================
def predict_requirement(text, model, tokenizer, id2label, show_top_k=3):
    """
    Predict the NFR category for a given requirement text

    Args:
        text: Requirement text to classify
        model: Trained model
        tokenizer: RoBERTa tokenizer
        id2label: Dictionary mapping class indices to labels
        show_top_k: Number of top predictions to show

    Returns:
        Dictionary with prediction results
    """
    model.eval()

    # Tokenize input
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=config.MAX_LENGTH,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(config.DEVICE)
    attention_mask = encoding['attention_mask'].to(config.DEVICE)

    # Predict
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=-1)
        confidence, predicted = torch.max(probs, 1)

    # Get top-k predictions
    topk_probs, topk_indices = torch.topk(probs[0], min(show_top_k, len(id2label)))

    # Format results
    predicted_class = id2label[predicted.cpu().item()]
    confidence_score = confidence.cpu().item()

    top_predictions = [
        {
            'class': id2label[idx.cpu().item()],
            'confidence': prob.cpu().item()
        }
        for prob, idx in zip(topk_probs, topk_indices)
    ]

    return {
        'predicted_class': predicted_class,
        'confidence': confidence_score,
        'top_predictions': top_predictions
    }


# ============================================================================
# INTERACTIVE PREDICTION MODE
# ============================================================================
def interactive_mode():
    """Interactive prediction with detailed output"""

    # Load model components
    model, tokenizer, id2label = load_model()

    print("\n" + "=" * 80)
    print("INTERACTIVE PREDICTION MODE")
    print("=" * 80)
    print("Enter requirement text to classify (or 'quit' to exit)")
    print("=" * 80 + "\n")

    while True:
        user_input = input("📝 Requirement: ").strip()

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Exiting prediction mode...")
            break

        if not user_input:
            continue

        # Get prediction
        result = predict_requirement(user_input, model, tokenizer, id2label)

        # Display results
        print(f"\n✅ Prediction: {result['predicted_class']}")
        print(f"📊 Confidence: {result['confidence']:.4f} ({result['confidence']*100:.2f}%)")

        # Show warning for low confidence
        if result['confidence'] < 0.85:
            print("\n⚠️ Low confidence! Top 3 predictions:")
        else:
            print("\n📈 Top 3 predictions:")

        for i, pred in enumerate(result['top_predictions'], 1):
            bar_length = int(pred['confidence'] * 40)
            bar = "█" * bar_length + "░" * (40 - bar_length)
            print(f"  {i}. {pred['class']:<20} {bar} {pred['confidence']:.4f}")

        print()


# ============================================================================
# BATCH PREDICTION FUNCTION
# ============================================================================
def predict_batch(texts, model, tokenizer, id2label):
    """
    Predict NFR categories for multiple requirements

    Args:
        texts: List of requirement texts
        model: Trained model
        tokenizer: RoBERTa tokenizer
        id2label: Dictionary mapping class indices to labels

    Returns:
        List of prediction dictionaries
    """
    results = []

    for text in texts:
        result = predict_requirement(text, model, tokenizer, id2label)
        results.append(result)

    return results


# ============================================================================
# MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    # Start interactive mode
    interactive_mode()

    # Example: Batch prediction
    # model, tokenizer, id2label = load_model()
    #
    # test_requirements = [
    #     "The system shall respond to user queries within 2 seconds",
    #     "The application must be compatible with iOS 14 and above",
    #     "User passwords must be encrypted using AES-256"
    # ]
    #
    # results = predict_batch(test_requirements, model, tokenizer, id2label)
    #
    # for req, result in zip(test_requirements, results):
    #     print(f"\nRequirement: {req}")
    #     print(f"Predicted: {result['predicted_class']} ({result['confidence']:.4f})")

LOADING NFR CLASSIFICATION MODEL
✅ Loaded 7 classes: Compatibility, Maintainability, Performance, Portability, Reliability, Security, Usability
✅ Loaded tokenizer: roberta-base


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Loaded model from epoch 8 with Val F1: 0.9710
🖥️ Using device: cuda

INTERACTIVE PREDICTION MODE
Enter requirement text to classify (or 'quit' to exit)

📝 Requirement: The system shall ensure all data communication is encrypted using HTTPS protocol.

✅ Prediction: Security
📊 Confidence: 0.8794 (87.94%)

📈 Top 3 predictions:
  1. Security             ███████████████████████████████████░░░░░ 0.8794
  2. Reliability          █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.0281
  3. Usability            ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.0229

📝 Requirement: The system shall achieve at least 80% unit test coverage to maintain code quality.

✅ Prediction: Maintainability
📊 Confidence: 0.8730 (87.30%)

📈 Top 3 predictions:
  1. Maintainability      ██████████████████████████████████░░░░░░ 0.8730
  2. Portability          █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.0260
  3. Security             ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░ 0.0242

📝 Requirement: The system shall automatically